In [1]:
%load_ext autoreload
%autoreload 2
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:95% !important; }</style>"))
import warnings
warnings.filterwarnings("ignore")

import os
import pandas as pd

ddir = 'data/'

df = pd.read_csv(os.path.join(ddir,'df_samp_20to30_10000.csv'))
df.head(5)

,smiles
0,COc1cccc2oc(=O)cc(N3CCOCC3)c12
1,CNCCCCNC1(c2cc3ccccc3s2)CCCCC1
2,NCC1OC2OCCC1(O)C2O
3,Cc1c(O)c(CN)c(Cl)c(C)c1Cl
4,CCCC1C2CNCC12c1ccc(Cl)c(Cl)c1


In [2]:
from rdkit.Chem import PandasTools
def displaydf(df):
    return HTML(df.to_html(notebook=True))

PandasTools.AddMoleculeColumnToFrame(df,'smiles','mol',includeFingerprints=False)

from data_utils import *
atom_to_cnt = get_atom_cnts(df.smiles)
# atom_to_cnt

In [3]:
from rdkit_utils import *
from graph_augs import *

from rdkit import RDLogger  
RDLogger.DisableLog('rdApp.*')

def get_augs(smiles,maximum=10):
    
    mol = Chem.MolFromSmiles(smiles)

    idc = [i for i in range(0,(mol.GetNumAtoms()))]
    random.shuffle(idc)
    
    goods = 0 
    aug_smiles = []
    for i in idc:
        if goods==maximum:
            break
            
        atom_type = get_weighted_random_atom(atom_to_cnt)
        
        try: 
            mol_aug = add_atom_to_mol(mol, atom_type, i, clean_aroms = True)
            if mol_aug.GetNumAtoms()==0:
                continue
            else:
                sm = Chem.MolToSmiles(mol_aug)
                goods+=1
                aug_smiles.append(sm)
        except Exception as e:
            continue
            
    return (smiles, aug_smiles)

from joblib import Parallel, delayed
import multiprocessing

parallelizer = Parallel(n_jobs=multiprocessing.cpu_count()-1, backend= 'multiprocessing' )
augs_tasks = (delayed(get_augs)(smiles) for smiles in df.smiles)
smiles_to_augs = parallelizer(augs_tasks)

smiles_to_augs = {k:v for k,v in smiles_to_augs}

In [18]:
import csv

with open('data/augmented_20to30_10000.csv','w') as out_file:
    writer = csv.writer(out_file, delimiter=',')
    for k,v in smiles_to_augs.items():
        line = [k] + v
        writer.writerow(line)

In [19]:
# Test opening it !!!

fname = os.path.join(ddir,'augmented_20to30_10000.csv')

test_dict = {}
with open(fname) as csvfile:
    reader = csv.reader(csvfile)
    for row in reader:
        test_dict[row[0]] = row[1:]